In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder 
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("C:\\Users\\hello\\Desktop\\AIML\\project1\\HR_Employee_Attrition.csv.csv")
print(df.shape)
print(df["Attrition"].value_counts(normalize=True))

In [ ]:
#Income vs Attrition
plt.figure(figsize=(10, 8))
sns.boxplot(x="Attrition", y ="MonthlyIncome", data=df)
plt.title("Income vs Attrition")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")
plt.show()

In [ ]:
#OverTime vs Attrition
plt.figure(figsize=(10, 8))
sns.countplot(x = "OverTime", hue="Attrition", data=df)
plt.title("OverTime vs Attrition")
plt.xlabel("OverTime")

In [ ]:
#Drop irrelevant columns
df.drop(["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"], axis=1, inplace=True)
#df.info()

#Encode categorical Variables (conver datatype into numeric value)
encoder = {}
for col in df.select_dtypes(include=["object"]).columns:
    lb = LabelEncoder()
    df[col] = lb.fit_transform(df[col])
    encoder[col] = lb

    #split data into features and target 
    x = df.drop("Attrition", axis=1)
    y = df["Attrition"] 

    # split data into training and testing sets
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=42)

In [ ]:
#Define the model
rf = RandomForestClassifier(random_state=42)

#Hyperparameter tuning
param_grid = {
    "n_estimators": [50,100],
    "max_depth":[10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1')
grid_search.fit(x_train, y_train)
print("Best Hyperparameters:", grid_search.best_params_)

In [ ]:
best_model = grid_search.best_estimator_
predictions = best_model.predict(x_test)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

importances =best_model.feature_importances_
feature_importance_df =pd.DataFrame({
    "Feature":x.columns,
    "Importance":importances
}).sort_values(by="Importance",ascending=False)
plt.figure(figsize=(12, 8))
sns.barplot(x="Importance",y="Feature",data=feature_importance_df)
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")

In [ ]:
model = RandomForestClassifier(**grid_search.best_params_,random_state=42)
model.fit(x, y)
#Save the model
joblib.dump(model, "employee_attrition_model.pkl")
joblib.dump(encoder, "label_encoder.pkl")
joblib.dump(x.columns.to_list(), "feature_columns.pkl")